# 🎙️ Speech Quality Evaluation: Natural vs. Synthetic
## Disagreement-Aware Quality Score (DAQS)

| Field | Value |
|---|---|
| **Student** | Qamar Mehmood |
| **Roll No** | 26I-7800 |
| **Topic** | T6 — Speech Quality Evaluation: Natural vs. Synthetic |
| **Course** | MSCS – Machine Learning, FAST-NUCES Islamabad |
| **Novel Contribution** | Disagreement-Aware Quality Score (DAQS) |

---
### Project Structure
1. Environment Setup
2. Data Preparation (LJSpeech + 3 TTS systems)
3. Metric Computation (MCD, PESQ, UTMOS, DNSMOS, WER)
4. Normalization & EDA
5. Disagreement Analysis *(T6-c requirement)*
6. DAQS Implementation *(T6-d — Novel Contribution)*
7. Baseline Comparison
8. Statistical Significance Testing
9. Ablation Study
10. Dataset Documentation


---
## Section 1: Environment Setup
> ⚠️ **Run Cell 1.1 once, then restart kernel, then run all remaining cells.**

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1.1 — Install all dependencies (run ONCE, then restart kernel)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

packages = [
    "TTS",               # Coqui TTS — 3 synthesis systems
    "pymcd",             # Mel Cepstral Distortion
    "pesq",              # Perceptual Evaluation of Speech Quality
    "openai-whisper",    # WER via ASR
    "speechmos",         # DNSMOS (Microsoft neural MOS)
    "torch",
    "torchaudio",
    "librosa",
    "soundfile",
    "matplotlib",
    "seaborn",
    "pandas",
    "scipy",
    "scikit-learn",
    "tqdm",
    "tabulate",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                   capture_output=True)

print("\n✅ All packages installed.")
print("   ➡️  Now: Kernel → Restart Kernel, then run all remaining cells.")


Installing TTS...
Installing pymcd...
Installing pesq...
Installing openai-whisper...
Installing speechmos...
Installing torch...
Installing torchaudio...
Installing librosa...
Installing soundfile...
Installing matplotlib...
Installing seaborn...
Installing pandas...
Installing scipy...
Installing scikit-learn...
Installing tqdm...
Installing tabulate...

✅ All packages installed.
   ➡️  Now: Kernel → Restart Kernel, then run all remaining cells.


---
## Section 2: Imports & Configuration

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.1 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import os, json, warnings, subprocess
import onnxruntime  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
import librosa
import soundfile as sf
from scipy import stats
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from tabulate import tabulate

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False})

print(f"✅ Imports OK")
print(f"   PyTorch : {torch.__version__}")
print(f"   CUDA    : {torch.cuda.is_available()} (GPU not required — CPU is fine)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


✅ Imports OK
   PyTorch : 2.3.0+cpu
   CUDA    : False (GPU not required — CPU is fine)


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.2 — Configuration  ⚠️ CHANGE LJSPEECH_PATH BELOW ⚠️
# ─────────────────────────────────────────────────────────────────────────────

# LJSPEECH_PATH = r"C:\datasets\LJSpeech-1.1"   # <── CHANGE THIS to your path
LJSPEECH_PATH = r"D:\\MSCS\Semester 1\\ML\\Project\\LJSpeech-1.1"

N_UTTERANCES  = 100       # utterances per system (100 is fast; increase to 200 for more data)
SAMPLE_RATE   = 16000     # all audio resampled to 16 kHz mono
LAMBDA_RANGE  = np.round(np.arange(0.0, 2.1, 0.1), 1)   # λ search space

OUTPUT_DIR = Path("./daqs_output")
for sub in ["audio/natural", "audio/speedy_tts", "audio/tacotron2", "audio/vits",
            "metrics", "plots", "results"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

SYSTEMS = ["natural", "speedy_tts", "tacotron2", "vits"]
SYSTEM_LABELS = {
    "natural"    : "Natural (LJSpeech)",
    "speedy_tts" : "SpeedySpeech (concat-proxy)",
    "tacotron2"  : "Tacotron2-DDC (neural)",
    "vits"       : "VITS (SOTA)",
}
SYSTEM_COLORS = {
    "natural"    : "#2196F3",
    "speedy_tts" : "#F44336",
    "tacotron2"  : "#FF9800",
    "vits"       : "#4CAF50",
}

print(f"✅ Config ready | Output: {OUTPUT_DIR.resolve()}")
print(f"   LJSpeech path : {LJSPEECH_PATH}")
print(f"   Utterances    : {N_UTTERANCES} per system  ({N_UTTERANCES * 4} total)")


✅ Config ready | Output: D:\MSCS\Semester 1\ML\Project\daqs_output
   LJSpeech path : D:\\MSCS\Semester 1\\ML\\Project\\LJSpeech-1.1
   Utterances    : 100 per system  (400 total)


---
## Section 3: Data Preparation

**Dataset:** LJSpeech-1.1 (natural speech) + 3 TTS systems synthesised on same text prompts.  
**Why this approach:** By generating all TTS systems on *identical* text, reference-based metrics (MCD, PESQ) can compare audio directly to the natural recording.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.1 — Load LJSpeech metadata & select N utterances
# ─────────────────────────────────────────────────────────────────────────────
meta_path = Path(LJSPEECH_PATH) / "metadata.csv"
assert meta_path.exists(), (
    f"❌ metadata.csv not found at {meta_path}\n"
    f"   Make sure LJSPEECH_PATH points to the extracted LJSpeech-1.1 folder."
)

df_meta = pd.read_csv(meta_path, sep="|", header=None,
                       names=["id", "text", "normalized_text"], quoting=3)

# Drop any rows with missing normalized text
df_meta = df_meta.dropna(subset=["normalized_text"]).reset_index(drop=True)
df_sample = df_meta.iloc[:N_UTTERANCES].copy().reset_index(drop=True)
df_sample["wav_path"] = df_sample["id"].apply(
    lambda x: str(Path(LJSPEECH_PATH) / "wavs" / f"{x}.wav")
)

missing = [r["id"] for _, r in df_sample.iterrows() if not Path(r["wav_path"]).exists()]
assert len(missing) == 0, f"❌ {len(missing)} wav files missing: {missing[:5]}"

print(f"✅ Loaded {len(df_sample)} utterances from LJSpeech")
print(f"   Avg text length: {df_sample.normalized_text.str.len().mean():.0f} chars")
print()
print(df_sample[["id", "normalized_text"]].head(5).to_string(index=False))


✅ Loaded 100 utterances from LJSpeech
   Avg text length: 100 chars

        id                                                                                                                                             normalized_text
LJ001-0001     Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
LJ001-0002                                                                                                                              in being comparatively modern.
LJ001-0003 For although the Chinese took impressions from wood blocks engraved in relief for centuries before the woodcutters of the Netherlands, by a similar process
LJ001-0004                                                                   produced the block books, which were the immediate predecessors of the true printed book,
LJ001-0005             the invention of movable metal letters in the middle of the fifteenth cen

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.2 — Resample natural audio to 16 kHz mono
# ─────────────────────────────────────────────────────────────────────────────
nat_dir = OUTPUT_DIR / "audio" / "natural"
skipped = 0

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Resampling natural audio"):
    out = nat_dir / f"{row['id']}.wav"
    if out.exists():
        skipped += 1
        continue
    wav, sr = torchaudio.load(row["wav_path"])
    wav = wav.mean(dim=0, keepdim=True)            # stereo → mono
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    torchaudio.save(str(out), wav, SAMPLE_RATE)

n_files = len(list(nat_dir.glob("*.wav")))
print(f"✅ Natural audio ready: {n_files} files (skipped {skipped} already done)")


Resampling natural audio: 100%|██████████| 100/100 [00:00<00:00, 4861.66it/s]

✅ Natural audio ready: 100 files (skipped 100 already done)


### TTS Generation
> ⚠️ **First run downloads ~400 MB of model weights and generates 300 audio files.**  
> Progress bars show status. Files are cached — re-runs skip existing files automatically.  
> Expected time: **20–60 min on CPU** (depends on machine speed).


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.3 — TTS System 1: SpeedySpeech (fast, less natural — concatenative proxy)
# ─────────────────────────────────────────────────────────────────────────────
from TTS.api import TTS as CoquiTTS

sys1_dir = OUTPUT_DIR / "audio" / "speedy_tts"
existing = len(list(sys1_dir.glob("*.wav")))
todo = [r for _, r in df_sample.iterrows() if not (sys1_dir / f"{r['id']}.wav").exists()]

if len(todo) == 0:
    print(f"✅ SpeedySpeech already done ({existing} files cached)")
else:
    print(f"Loading SpeedySpeech model (downloads on first run)...")
    try:
        tts1 = CoquiTTS(model_name="tts_models/en/ljspeech/speedy-speech", progress_bar=False)
        for row in tqdm(todo, desc="SpeedySpeech synthesis"):
            out = sys1_dir / f"{row['id']}.wav"
            tts1.tts_to_file(text=row["normalized_text"], file_path=str(out))
        print(f"✅ SpeedySpeech done: {len(list(sys1_dir.glob('*.wav')))} files")
    except Exception as e:
        print(f"⚠️  SpeedySpeech model failed: {e}")
        print("   Trying fallback: glow-tts...")
        try:
            tts1 = CoquiTTS(model_name="tts_models/en/ljspeech/glow-tts", progress_bar=False)
            for row in tqdm(todo, desc="GlowTTS fallback"):
                out = sys1_dir / f"{row['id']}.wav"
                tts1.tts_to_file(text=row["normalized_text"], file_path=str(out))
            print(f"✅ GlowTTS fallback done: {len(list(sys1_dir.glob('*.wav')))} files")
        except Exception as e2:
            print(f"❌ Both models failed: {e2}")


✅ SpeedySpeech already done (100 files cached)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.4 — TTS System 2: Tacotron2-DDC (classical neural TTS)
# ─────────────────────────────────────────────────────────────────────────────
sys2_dir = OUTPUT_DIR / "audio" / "tacotron2"
todo2 = [r for _, r in df_sample.iterrows() if not (sys2_dir / f"{r['id']}.wav").exists()]

if len(todo2) == 0:
    print(f"✅ Tacotron2 already done ({len(list(sys2_dir.glob('*.wav')))} files cached)")
else:
    print("Loading Tacotron2-DDC model (downloads on first run)...")
    try:
        tts2 = CoquiTTS(model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False)
        for row in tqdm(todo2, desc="Tacotron2-DDC synthesis"):
            out = sys2_dir / f"{row['id']}.wav"
            tts2.tts_to_file(text=row["normalized_text"], file_path=str(out))
        print(f"✅ Tacotron2 done: {len(list(sys2_dir.glob('*.wav')))} files")
    except Exception as e:
        print(f"❌ Tacotron2 failed: {e}")


✅ Tacotron2 already done (100 files cached)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.5 — TTS System 3: VITS (state-of-the-art end-to-end TTS)
# ─────────────────────────────────────────────────────────────────────────────
sys3_dir = OUTPUT_DIR / "audio" / "vits"
todo3 = [r for _, r in df_sample.iterrows() if not (sys3_dir / f"{r['id']}.wav").exists()]

if len(todo3) == 0:
    print(f"✅ VITS already done ({len(list(sys3_dir.glob('*.wav')))} files cached)")
else:
    print("Loading VITS model (downloads on first run)...")
    try:
        tts3 = CoquiTTS(model_name="tts_models/en/ljspeech/vits", progress_bar=False)
        for row in tqdm(todo3, desc="VITS (SOTA) synthesis"):
            out = sys3_dir / f"{row['id']}.wav"
            tts3.tts_to_file(text=row["normalized_text"], file_path=str(out))
        print(f"✅ VITS done: {len(list(sys3_dir.glob('*.wav')))} files")
    except Exception as e:
        print(f"❌ VITS failed: {e}")


✅ VITS already done (100 files cached)


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.6 — Resample all TTS outputs to 16 kHz mono
# (TTS models may output at 22050 Hz — normalize everything)
# ─────────────────────────────────────────────────────────────────────────────
for sys_name in ["speedy_tts", "tacotron2", "vits"]:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    files   = list(sys_dir.glob("*.wav"))
    resampled = 0
    for f in tqdm(files, desc=f"Normalize {sys_name}"):
        wav, sr = torchaudio.load(str(f))
        wav = wav.mean(dim=0, keepdim=True)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
            torchaudio.save(str(f), wav, SAMPLE_RATE)
            resampled += 1
    print(f"   {sys_name}: {len(files)} files, {resampled} resampled to {SAMPLE_RATE} Hz")

print("\n✅ All audio at 16 kHz mono")


Normalize speedy_tts: 100%|██████████| 100/100 [00:00<00:00, 459.28it/s]


   speedy_tts: 100 files, 0 resampled to 16000 Hz


Normalize tacotron2: 100%|██████████| 100/100 [00:00<00:00, 926.84it/s]


   tacotron2: 100 files, 0 resampled to 16000 Hz


Normalize vits: 100%|██████████| 100/100 [00:00<00:00, 1093.93it/s]

   vits: 100 files, 0 resampled to 16000 Hz

✅ All audio at 16 kHz mono


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.7 — Build master file index
# ─────────────────────────────────────────────────────────────────────────────
records = []
for sys_name in SYSTEMS:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in df_sample.iterrows():
        f = sys_dir / f"{row['id']}.wav"
        if f.exists():
            records.append({
                "id"      : row["id"],
                "text"    : row["normalized_text"],
                "system"  : sys_name,
                "wav_path": str(f),
                "label"   : 1 if sys_name == "natural" else 0,
            })

df_files = pd.DataFrame(records)
print(f"✅ Master index built: {len(df_files)} total utterances")
print()
print(df_files.groupby("system")["id"].count().rename("count").to_frame().to_string())
print()
print("⚠️  If any system shows < 100, re-run its TTS cell above.")


✅ Master index built: 400 total utterances

            count
system           
natural       100
speedy_tts    100
tacotron2     100
vits          100

⚠️  If any system shows < 100, re-run its TTS cell above.


---
## Section 4: Metric Computation

Computing **5 objective metrics** as required by T6(b):

| Metric | Type | Higher = better? | Notes |
|---|---|---|---|
| MCD | Reference-based | ❌ lower | Mel Cepstral Distortion |
| PESQ | Reference-based | ✅ higher | Perceptual quality (ITU-T P.862) |
| UTMOS | Reference-free | ✅ higher | Neural MOS predictor |
| DNSMOS | Reference-free | ✅ higher | Microsoft DNS neural MOS |
| WER | Reference-based | ❌ lower | Word Error Rate via Whisper ASR |

Natural speech uses MCD=0, PESQ=4.5 (max), WER=0 (ideal) with itself as reference.


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.1 — Metric 1: MCD (Mel Cepstral Distortion)
# Lower MCD = more similar to natural. We later invert so higher = better.
# ─────────────────────────────────────────────────────────────────────────────
from pymcd.mcd import Calculate_MCD

mcd_calc = Calculate_MCD(MCD_mode="dtw_sl")   # DTW with slope constraint

def compute_mcd(ref_path, syn_path):
    try:
        val = mcd_calc.calculate_mcd(ref_path, syn_path)
        return float(val)
    except Exception:
        return np.nan

nat_dir  = OUTPUT_DIR / "audio" / "natural"
mcd_rows = []

# Synthetic systems vs natural reference
for sys_name in ["speedy_tts", "tacotron2", "vits"]:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"MCD {sys_name}"):
        ref = nat_dir / f"{row['id']}.wav"
        syn = sys_dir / f"{row['id']}.wav"
        if ref.exists() and syn.exists():
            mcd_rows.append({"id": row["id"], "system": sys_name,
                             "mcd": compute_mcd(str(ref), str(syn))})

# Natural: MCD with itself = 0 (perfect)
for _, row in df_sample.iterrows():
    mcd_rows.append({"id": row["id"], "system": "natural", "mcd": 0.0})

df_mcd = pd.DataFrame(mcd_rows)
print("✅ MCD computed")
print(df_mcd.groupby("system")["mcd"].agg(["mean","std","min","max"]).round(2).to_string())


MCD vits: 100%|██████████| 100/100 [02:08<00:00,  1.28s/it]

✅ MCD computed
             mean    std   min     max
system                                
natural      0.00   0.00  0.00    0.00
speedy_tts   7.11   0.87  5.34    9.35
tacotron2   10.40  21.08  5.27  159.28
vits         5.86   0.80  4.10    8.12


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.2 — Metric 2: PESQ (Perceptual Evaluation of Speech Quality)
# Wideband mode ('wb') requires 16 kHz. Range: -0.5 to 4.5 (higher=better).
# ─────────────────────────────────────────────────────────────────────────────
from pesq import pesq as pesq_fn

def compute_pesq(ref_path, deg_path, sr=16000):
    try:
        ref, _ = librosa.load(ref_path, sr=sr, mono=True)
        deg, _ = librosa.load(deg_path, sr=sr, mono=True)
        min_len = min(len(ref), len(deg))
        if min_len < int(sr * 0.1):      # skip utterances < 100ms
            return np.nan
        ref, deg = ref[:min_len], deg[:min_len]
        return float(pesq_fn(sr, ref, deg, "wb"))
    except Exception:
        return np.nan

pesq_rows = []
for sys_name in ["speedy_tts", "tacotron2", "vits"]:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"PESQ {sys_name}"):
        ref = nat_dir / f"{row['id']}.wav"
        deg = sys_dir / f"{row['id']}.wav"
        if ref.exists() and deg.exists():
            pesq_rows.append({"id": row["id"], "system": sys_name,
                              "pesq": compute_pesq(str(ref), str(deg))})

# Natural: PESQ with itself = 4.5 (maximum possible)
for _, row in df_sample.iterrows():
    pesq_rows.append({"id": row["id"], "system": "natural", "pesq": 4.5})

df_pesq = pd.DataFrame(pesq_rows)
print("✅ PESQ computed")
print(df_pesq.groupby("system")["pesq"].agg(["mean","std","min","max"]).round(3).to_string())


PESQ vits: 100%|██████████| 100/100 [00:33<00:00,  2.95it/s]

✅ PESQ computed
             mean    std    min    max
system                                
natural     4.500  0.000  4.500  4.500
speedy_tts  1.075  0.030  1.036  1.180
tacotron2   1.091  0.071  1.033  1.464
vits        1.114  0.055  1.041  1.399


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.3 — Metric 3: UTMOS (Utterance-level TTS MOS predictor)
# Reference-free neural MOS predictor. Higher = better predicted MOS.
# ─────────────────────────────────────────────────────────────────────────────
print("Loading UTMOS model via torch.hub (downloads ~80 MB on first run)...")

utmos_model = None
try:
    utmos_model = torch.hub.load(
        "tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True
    )
    utmos_model.eval()
    print("✅ UTMOS model loaded")
except Exception as e:
    print(f"⚠️  UTMOS load failed: {e}")
    print("   UTMOS will be NaN — all other metrics still computed.")

def compute_utmos(wav_path, model, sr=SAMPLE_RATE):
    if model is None:
        return np.nan
    try:
        wav, orig_sr = torchaudio.load(wav_path)
        if orig_sr != sr:
            wav = torchaudio.functional.resample(wav, orig_sr, sr)
        wav = wav.mean(0, keepdim=True)
        with torch.no_grad():
            score = model(wav, sr)
        return float(score.item())
    except Exception:
        return np.nan

utmos_rows = []
for sys_name in SYSTEMS:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"UTMOS {sys_name}"):
        f = sys_dir / f"{row['id']}.wav"
        if f.exists():
            utmos_rows.append({"id": row["id"], "system": sys_name,
                               "utmos": compute_utmos(str(f), utmos_model)})

df_utmos = pd.DataFrame(utmos_rows)
nan_count = df_utmos["utmos"].isna().sum()
print(f"✅ UTMOS computed ({nan_count} NaN — expected if model unavailable)")
print(df_utmos.groupby("system")["utmos"].agg(["mean","std"]).round(3).to_string())


Using cache found in C:\Users\qamar/.cache\torch\hub\tarepan_SpeechMOS_v1.2.0


Loading UTMOS model via torch.hub (downloads ~80 MB on first run)...
✅ UTMOS model loaded


UTMOS vits: 100%|██████████| 100/100 [01:49<00:00,  1.10s/it]

✅ UTMOS computed (0 NaN — expected if model unavailable)
             mean    std
system                  
natural     4.362  0.109
speedy_tts  3.924  0.281
tacotron2   4.162  0.465
vits        4.375  0.159


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.4 — Metric 4: DNSMOS (Microsoft DNS MOS — reference-free neural MOS)
# Uses OVRL (overall) score. Higher = better.
# ─────────────────────────────────────────────────────────────────────────────

import sys
sys.path = [p for p in sys.path if "tarepan_SpeechMOS" not in p]

if "speechmos" in sys.modules:
    del sys.modules["speechmos"]

try:
    from speechmos import dnsmos as dns_mod
    HAS_SPEECHMOS = True
    print("✅ speechmos loaded")
except ImportError:
    HAS_SPEECHMOS = False
    print("⚠️  speechmos not found. Trying install...")
    subprocess.run([sys.executable, "-m", "pip", "install", "speechmos", "-q"])
    try:
        from speechmos import dnsmos as dns_mod
        HAS_SPEECHMOS = True
        print("✅ speechmos installed and loaded")
    except Exception as e:
        HAS_SPEECHMOS = False
        print("❌ DNSMOS unavailable — will be NaN")
        import traceback
        traceback.print_exc()
        print(f"Details: {e}")
        import sys; print("Kernel python:", sys.executable)

def compute_dnsmos(wav_path, sr=SAMPLE_RATE):
    if not HAS_SPEECHMOS:
        return np.nan
    try:
        wav, _ = librosa.load(wav_path, sr=sr, mono=True)
        result  = dns_mod.run(wav, sr)
        return float(result["ovrl_mos"])
    except Exception as e:
        return np.nan

dnsmos_rows = []
for sys_name in SYSTEMS:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"DNSMOS {sys_name}"):
        f = sys_dir / f"{row['id']}.wav"
        if f.exists():
            dnsmos_rows.append({"id": row["id"], "system": sys_name,
                                "dnsmos": compute_dnsmos(str(f))})

df_dnsmos = pd.DataFrame(dnsmos_rows)
print("✅ DNSMOS computed")
print(df_dnsmos.groupby("system")["dnsmos"].agg(["mean","std","min","max"]).round(3).to_string())


✅ speechmos loaded


DNSMOS vits: 100%|██████████| 100/100 [01:08<00:00,  1.46it/s]

✅ DNSMOS computed
             mean    std    min    max
system                                
natural     3.239  0.188  2.538  3.518
speedy_tts  3.315  0.127  2.958  3.552
tacotron2   3.277  0.275  1.840  3.577
vits        3.276  0.155  2.714  3.542


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.5 — Metric 5: WER via Whisper-small ASR
# WER = word error rate (lower = more intelligible). We invert later (higher=better).
# ─────────────────────────────────────────────────────────────────────────────
import whisper
from difflib import SequenceMatcher

print("Loading Whisper-small (downloads ~460 MB on first run)...")
asr = whisper.load_model("small")
print("✅ Whisper-small loaded")

def compute_wer(wav_path, reference_text):
    try:
        result    = asr.transcribe(str(wav_path), language="en", fp16=False)
        hyp_words = result["text"].strip().lower().split()
        ref_words = reference_text.strip().lower().split()
        if len(ref_words) == 0:
            return np.nan
        matcher   = SequenceMatcher(None, ref_words, hyp_words)
        wer_val   = 1.0 - matcher.ratio()
        return float(np.clip(wer_val, 0.0, 1.0))
    except Exception:
        return np.nan

wer_rows = []
for sys_name in SYSTEMS:
    sys_dir = OUTPUT_DIR / "audio" / sys_name
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=f"WER {sys_name}"):
        f = sys_dir / f"{row['id']}.wav"
        if f.exists():
            wer_rows.append({"id": row["id"], "system": sys_name,
                             "wer": compute_wer(str(f), row["normalized_text"])})

df_wer = pd.DataFrame(wer_rows)
print("✅ WER computed")
print(df_wer.groupby("system")["wer"].agg(["mean","std","min","max"]).round(3).to_string())


Loading Whisper-small (downloads ~460 MB on first run)...
✅ Whisper-small loaded


WER vits: 100%|██████████| 100/100 [09:15<00:00,  5.56s/it]

✅ WER computed
             mean    std  min  max
system                            
natural     0.119  0.127  0.0  1.0
speedy_tts  0.179  0.133  0.0  1.0
tacotron2   0.159  0.162  0.0  1.0
vits        0.137  0.127  0.0  1.0


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.6 — Assemble all metrics into one dataframe
# ─────────────────────────────────────────────────────────────────────────────
df_all = df_files.copy()

for metric_df, col in [(df_mcd,"mcd"), (df_pesq,"pesq"),
                        (df_utmos,"utmos"), (df_dnsmos,"dnsmos"), (df_wer,"wer")]:
    df_all = df_all.merge(metric_df[["id","system",col]], on=["id","system"], how="left")

df_all.to_csv(OUTPUT_DIR / "metrics" / "all_metrics_raw.csv", index=False)

print("✅ All metrics assembled")
print(f"   Shape: {df_all.shape}")
print()
print("Mean metrics per system (raw values):")
print(df_all.groupby("system")[["mcd","pesq","utmos","dnsmos","wer"]].mean().round(3).to_string())
print()
nan_summary = df_all[["mcd","pesq","utmos","dnsmos","wer"]].isna().sum()
print("NaN counts (0 = fully computed):")
print(nan_summary.to_string())


✅ All metrics assembled
   Shape: (400, 10)

Mean metrics per system (raw values):
               mcd   pesq  utmos  dnsmos    wer
system                                         
natural      0.000  4.500  4.362   3.239  0.119
speedy_tts   7.112  1.075  3.924   3.315  0.179
tacotron2   10.399  1.091  4.162   3.277  0.159
vits         5.864  1.114  4.375   3.276  0.137

NaN counts (0 = fully computed):
mcd        0
pesq       1
utmos      0
dnsmos    77
wer        0


---
## Section 5: Metric Normalization & EDA

Normalizing all metrics to **[0, 1] where 1 = best quality**, before computing disagreement.
- MCD: lower is better → invert
- PESQ: scale from [−0.5, 4.5] → [0, 1]
- UTMOS, DNSMOS: already higher=better → scale by observed range
- WER: lower is better → invert


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.1 — Normalize all metrics to [0,1], higher = better
# ─────────────────────────────────────────────────────────────────────────────
df_norm = df_all.copy()

# MCD: invert and normalize (0 = worst/max MCD, 1 = best/0 MCD)
mcd_max = df_norm["mcd"].replace([np.inf], np.nan).quantile(0.99)
df_norm["mcd_n"] = 1.0 - (df_norm["mcd"].clip(0, mcd_max).fillna(mcd_max) / (mcd_max + 1e-8))

# PESQ: scale [-0.5, 4.5] → [0,1]
PESQ_MIN, PESQ_MAX = -0.5, 4.5
df_norm["pesq_n"] = (df_norm["pesq"].fillna(PESQ_MIN) - PESQ_MIN) / (PESQ_MAX - PESQ_MIN)

# UTMOS: scale by observed range
u_min = df_norm["utmos"].min()
u_max = df_norm["utmos"].max()
df_norm["utmos_n"] = (df_norm["utmos"].fillna(u_min) - u_min) / (u_max - u_min + 1e-8)

# DNSMOS: scale by observed range  
d_min = df_norm["dnsmos"].min()
d_max = df_norm["dnsmos"].max()
df_norm["dnsmos_n"] = (df_norm["dnsmos"].fillna(d_min) - d_min) / (d_max - d_min + 1e-8)

# WER: invert (0 = perfect, 1 = completely wrong → flip)
df_norm["wer_n"] = 1.0 - df_norm["wer"].fillna(1.0)

METRIC_COLS  = ["mcd_n", "pesq_n", "utmos_n", "dnsmos_n", "wer_n"]
METRIC_NAMES = {"mcd_n":"MCD", "pesq_n":"PESQ", "utmos_n":"UTMOS",
                "dnsmos_n":"DNSMOS", "wer_n":"WER"}
df_norm["label"] = (df_norm["system"] == "natural").astype(int)

df_norm.to_csv(OUTPUT_DIR / "metrics" / "normalized_metrics.csv", index=False)
print("✅ Normalization complete (all metrics: 0 = worst, 1 = best)")
print()
print("Mean normalized metrics per system:")
print(df_norm.groupby("system")[METRIC_COLS].mean().round(3).to_string())


✅ Normalization complete (all metrics: 0 = worst, 1 = best)

Mean normalized metrics per system:
            mcd_n  pesq_n  utmos_n  dnsmos_n  wer_n
system                                             
natural     1.000   1.000    0.954     0.806  0.881
speedy_tts  0.263   0.315    0.816     0.586  0.821
tacotron2   0.236   0.315    0.891     0.604  0.841
vits        0.392   0.323    0.959     0.670  0.863


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.2 — Figure 1: Metric distributions by system
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)

for ax, col in zip(axes, METRIC_COLS):
    for sys in SYSTEMS:
        vals = df_norm[df_norm.system == sys][col].dropna()
        ax.hist(vals, bins=14, alpha=0.65, label=SYSTEM_LABELS[sys],
                color=SYSTEM_COLORS[sys], density=True, edgecolor='white', linewidth=0.4)
    ax.set_title(METRIC_NAMES[col], fontsize=12, fontweight="bold")
    ax.set_xlabel("Normalized score (higher = better)", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.legend(fontsize=7, framealpha=0.7)

plt.suptitle("Figure 1: Normalized Metric Distributions by TTS System",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig1_metric_distributions.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 1 saved")


✅ Figure 1 saved


In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.3 — Figure 2: Inter-metric correlation heatmaps (one per system)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))

for ax, sys in zip(axes, SYSTEMS):
    sub  = df_norm[df_norm.system == sys][METRIC_COLS].rename(columns=METRIC_NAMES)
    corr = sub.corr()
    mask = np.zeros_like(corr, dtype=bool)
    np.fill_diagonal(mask, True)   # hide self-correlations
    sns.heatmap(corr, ax=ax, cmap="coolwarm", vmin=-1, vmax=1,
                annot=True, fmt=".2f", linewidths=0.5,
                cbar=True, square=True, mask=mask,
                annot_kws={"size": 9})
    ax.set_title(f"{SYSTEM_LABELS[sys]}", fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=8)

plt.suptitle("Figure 2: Inter-Metric Correlation (per System) High off-diagonal correlation → metrics agree; low → potential disagreement",
             fontsize=11, fontweight="bold", y=1.06)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig2_correlation_heatmaps.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 2 saved")


✅ Figure 2 saved


---
## Section 6: Disagreement Analysis (T6-c Requirement)

**Definition:** For each utterance u, compute the mean absolute pairwise difference across all metric pairs:

$$D(u) = \frac{2}{n(n-1)} \sum_{i < j} |m_i(u) - m_j(u)|$$

where n = 5 metrics, all normalized to [0,1]. D ∈ [0,1]; high D means the metrics are conflicting.

**High-disagreement utterances** (D > θ) are flagged and classified into error types.


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.1 — Compute disagreement D(u) for every utterance
# ─────────────────────────────────────────────────────────────────────────────
def pairwise_disagreement(row, cols=METRIC_COLS):
    vals = row[cols].values.astype(float)
    vals = vals[~np.isnan(vals)]            # drop NaN metrics
    if len(vals) < 2:
        return np.nan
    n     = len(vals)
    pairs = [(vals[i], vals[j]) for i in range(n) for j in range(i+1, n)]
    return np.mean([abs(a - b) for a, b in pairs])

df_norm["D"]  = df_norm.apply(pairwise_disagreement, axis=1)
df_norm["mu"] = df_norm[METRIC_COLS].mean(axis=1)

# Threshold θ = global mean + 1 std
theta = df_norm["D"].mean() + df_norm["D"].std()
df_norm["high_disagreement"] = (df_norm["D"] > theta).astype(bool)

total_high = df_norm["high_disagreement"].sum()
pct_high   = df_norm["high_disagreement"].mean() * 100

print(f"✅ Disagreement D(u) computed for all {len(df_norm)} utterances")
print(f"   θ (threshold) = {theta:.4f}  (μ + 1σ)")
print(f"   High-disagreement utterances: {total_high} ({pct_high:.1f}%)")
print()
print("Mean D(u) per system:")
print(df_norm.groupby("system")[["D","mu"]].mean().round(4).to_string())


✅ Disagreement D(u) computed for all 400 utterances
   θ (threshold) = 0.4700  (μ + 1σ)
   High-disagreement utterances: 58 (14.5%)

Mean D(u) per system:
                 D      mu
system                    
natural     0.1112  0.9282
speedy_tts  0.3926  0.5603
tacotron2   0.4237  0.5774
vits        0.3891  0.6412


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.2 — Figure 3: Disagreement analysis plots
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Boxplot of D(u) per system ───────────────────────────────────────
data_by_sys = [df_norm[df_norm.system == s]["D"].dropna().values for s in SYSTEMS]
bp = ax1.boxplot(data_by_sys, patch_artist=True,
                  medianprops=dict(color="black", linewidth=2))
for patch, sys in zip(bp["boxes"], SYSTEMS):
    patch.set_facecolor(SYSTEM_COLORS[sys])
    patch.set_alpha(0.7)
ax1.axhline(theta, color="purple", linestyle="--", linewidth=1.5,
            label=f"θ = {theta:.3f}")
ax1.set_xticks(range(1, len(SYSTEMS)+1))
ax1.set_xticklabels([SYSTEM_LABELS[s] for s in SYSTEMS], rotation=12, fontsize=9)
ax1.set_ylabel("Disagreement D(u)")
ax1.set_title("D(u) by TTS System", fontweight="bold")
ax1.legend(fontsize=9)

# ── Right: Quality–Disagreement scatter ────────────────────────────────────
for sys in SYSTEMS:
    sub = df_norm[df_norm.system == sys]
    ax2.scatter(sub["mu"], sub["D"], alpha=0.5, s=22,
                label=SYSTEM_LABELS[sys], color=SYSTEM_COLORS[sys])
ax2.axhline(theta, color="purple", linestyle="--", linewidth=1.5,
            label=f"θ = {theta:.3f}")
ax2.set_xlabel("Mean quality μ(u)")
ax2.set_ylabel("Disagreement D(u)")
ax2.set_title("Quality vs. Disagreement Space", fontweight="bold")
ax2.legend(fontsize=8)

plt.suptitle("Figure 3: Disagreement Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig3_disagreement.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 3 saved")


✅ Figure 3 saved


In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.3 — Error Type Classification (T6-c: ≥ 3 error types required)
#
# For each high-disagreement utterance, we identify WHICH pair of metrics
# conflicts most, and classify into one of 3 structured error types:
#
# Type A — Spectral-Prosody Conflict
#           MCD (low-level spectral) disagrees with UTMOS/DNSMOS (holistic neural)
#           → Audio sounds natural holistically but is spectrally distant
#
# Type B — Intelligibility-Quality Conflict
#           WER (intelligibility) disagrees with PESQ/UTMOS (overall quality)
#           → Audio is perceptually pleasing but hard to transcribe (or vice versa)
#
# Type C — Reference vs Reference-Free Conflict
#           PESQ/MCD (reference-based) disagrees with UTMOS/DNSMOS (reference-free)
#           → Human perceptual standard conflicts with learned neural MOS
# ─────────────────────────────────────────────────────────────────────────────
def classify_error_type(row, cols=METRIC_COLS):
    mcd_n, pesq_n, utmos_n, dnsmos_n, wer_n = [row[c] for c in cols]

    spectral_mean  = np.nanmean([mcd_n, pesq_n])       # reference-based
    holistic_mean  = np.nanmean([utmos_n, dnsmos_n])   # neural reference-free
    intell_score   = wer_n                              # intelligibility

    type_A = abs(mcd_n - np.nanmean([utmos_n, dnsmos_n]))   # spectral vs holistic
    type_B = abs(intell_score - np.nanmean([pesq_n, utmos_n]))  # intel vs quality
    type_C = abs(spectral_mean - holistic_mean)               # ref vs ref-free

    scores = {
        "Type A: Spectral-Prosody Conflict"         : type_A,
        "Type B: Intelligibility-Quality Conflict"  : type_B,
        "Type C: Reference vs Reference-Free"       : type_C,
    }
    return max(scores, key=scores.get)

high_d = df_norm[df_norm.high_disagreement].copy()
if len(high_d) > 0:
    high_d["error_type"] = high_d.apply(classify_error_type, axis=1)
    error_counts = high_d["error_type"].value_counts()

    print("📊 Error Type Classification (T6-c requirement):")
    print(f"   Total high-disagreement utterances: {len(high_d)}")
    print()
    for etype, cnt in error_counts.items():
        pct = cnt / len(high_d) * 100
        print(f"   {etype}: {cnt} ({pct:.1f}%)")

    high_d[["id","system","D","mu","error_type"]+METRIC_COLS].to_csv(
        OUTPUT_DIR / "metrics" / "high_disagreement_utterances.csv", index=False)
    print("\n✅ Error classification saved")
else:
    print("⚠️  No high-disagreement utterances found — lower θ or check metric computation.")
    error_counts = pd.Series(dtype=int)


📊 Error Type Classification (T6-c requirement):
   Total high-disagreement utterances: 58

   Type B: Intelligibility-Quality Conflict: 42 (72.4%)
   Type A: Spectral-Prosody Conflict: 15 (25.9%)
   Type C: Reference vs Reference-Free: 1 (1.7%)

✅ Error classification saved


In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.4 — Figure 4: Error type distribution
# ─────────────────────────────────────────────────────────────────────────────
if len(error_counts) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    bar_colors = ["#E91E63", "#FF5722", "#9C27B0"]

    # Overall frequency
    bars = ax1.bar(range(len(error_counts)), error_counts.values,
                   color=bar_colors[:len(error_counts)], edgecolor="black", linewidth=0.7)
    ax1.set_xticks(range(len(error_counts)))
    ax1.set_xticklabels([e.split(":")[0] for e in error_counts.index], fontsize=10)
    ax1.set_ylabel("Count")
    ax1.set_title("Error Type Frequencies (High-Disagreement Utterances)", fontweight="bold")
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x()+bar.get_width()/2, h+0.3, str(int(h)),
                 ha="center", fontsize=11, fontweight="bold")

    # Per-system breakdown
    try:
        pivot = high_d.groupby(["system","error_type"]).size().unstack(fill_value=0)
        pivot.plot(kind="bar", ax=ax2,
                   color=bar_colors[:len(pivot.columns)], edgecolor="black", linewidth=0.7)
        ax2.set_title("Error Types by TTS System", fontweight="bold")
        ax2.set_xlabel("System"); ax2.set_ylabel("Count")
        ax2.legend(fontsize=8, title="Error type", bbox_to_anchor=(1.01,1))
        ax2.tick_params(axis="x", rotation=15)
    except Exception:
        ax2.text(0.5, 0.5, "Insufficient data for breakdown",
                 ha="center", va="center", transform=ax2.transAxes)

    plt.suptitle("Figure 4: Disagreement Error Types (T6-c)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "fig4_error_types.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("✅ Figure 4 saved")


✅ Figure 4 saved


---
## Section 7: DAQS — Disagreement-Aware Quality Score (T6-d — Novel Contribution)

**Formal definition:**

$$\text{DAQS}(u) = \mu(u) - \lambda \cdot D(u) \cdot (1 - \mu(u))$$

where:
- $\mu(u)$ = mean of 5 normalized metrics (overall quality estimate)
- $D(u)$ = mean absolute pairwise disagreement across all metric pairs
- $\lambda$ = calibration constant, fit on validation set

**Key insight:** The penalty term $\lambda \cdot D \cdot (1-\mu)$ is **largest** when disagreement is high and mean quality is mid-range (0.3–0.7) — the most ambiguous zone.  
It **shrinks** near μ=0 (all agree: poor) or μ=1 (all agree: excellent).

**Why this is novel:** Existing composite scores (UTMOS, hybrid MOS) just do weighted averages — they treat disagreement as noise to be smoothed. DAQS treats it as a structured penalty signal.


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7.1 — DAQS formula + λ calibration via grid search
# ─────────────────────────────────────────────────────────────────────────────
def compute_daqs(mu, D, lam):
    """
    DAQS(u) = mu(u) - lambda * D(u) * (1 - mu(u))

    Parameters
    ----------
    mu  : float or array  Mean normalized quality across all metrics
    D   : float or array  Pairwise disagreement (mean absolute deviation)
    lam : float           Calibration constant (fit on validation split)
    """
    return np.array(mu) - lam * np.array(D) * (1.0 - np.array(mu))

# ── Calibrate λ on Tacotron2 vs Natural (validation pair) ─────────────────
df_cal = df_norm[df_norm.system.isin(["natural","tacotron2"])].dropna(subset=["D","mu"])
y_cal  = df_cal["label"].values

best_lam, best_auc = 0.0, 0.0
lambda_aucs = []

for lam in LAMBDA_RANGE:
    scores = compute_daqs(df_cal["mu"].values, df_cal["D"].values, lam)
    auc    = roc_auc_score(y_cal, scores)
    lambda_aucs.append(auc)
    if auc > best_auc:
        best_auc, best_lam = auc, lam

print(f"✅ λ calibration complete")
print(f"   Best λ = {best_lam:.1f}  (AUC = {best_auc:.4f} on validation set)")

# Apply DAQS to ALL utterances using best λ
df_norm["daqs"] = compute_daqs(df_norm["mu"].values, df_norm["D"].values, best_lam)

print()
print("DAQS scores per system:")
print(df_norm.groupby("system")[["mu","D","daqs"]].mean().round(4).to_string())


✅ λ calibration complete
   Best λ = 0.0  (AUC = 1.0000 on validation set)

DAQS scores per system:
                mu       D    daqs
system                            
natural     0.9282  0.1112  0.9282
speedy_tts  0.5603  0.3926  0.5603
tacotron2   0.5774  0.4237  0.5774
vits        0.6412  0.3891  0.6412


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7.2 — Figure 5: λ calibration curve
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(LAMBDA_RANGE, lambda_aucs, "o-", color="#1976D2", linewidth=2, markersize=5)
ax.axvline(best_lam, color="red", linestyle="--", linewidth=1.5,
           label=f"Best λ = {best_lam:.1f}  (AUC={best_auc:.4f})")
ax.fill_between(LAMBDA_RANGE, lambda_aucs,
                alpha=0.08, color="#1976D2")
ax.set_xlabel("λ (disagreement penalty weight)")
ax.set_ylabel("AUC-ROC (Natural vs. Tacotron2)")
ax.set_title("Figure 5: λ Calibration — AUC-ROC vs. Penalty Weight",
             fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig5_lambda_calibration.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 5 saved")


✅ Figure 5 saved


---
## Section 8: Baseline Comparison

In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8.1 — Compare DAQS against all baselines
# ─────────────────────────────────────────────────────────────────────────────
df_test = df_norm.dropna(subset=METRIC_COLS+["daqs"]).copy()
y_true  = df_test["label"].values

results = {}

# B1–B5: Individual metrics
for col, name in METRIC_NAMES.items():
    s    = df_test[col].values
    auc  = roc_auc_score(y_true, s)
    pred = (s > np.median(s)).astype(int)
    f1   = f1_score(y_true, pred, zero_division=0)
    results[name] = {"AUC-ROC": round(auc,4), "F1": round(f1,4)}

# B6: Equal-weight average (naive composite)
naive = df_test[METRIC_COLS].mean(axis=1).values
auc   = roc_auc_score(y_true, naive)
pred  = (naive > np.median(naive)).astype(int)
results["Naive Average (μ)"] = {
    "AUC-ROC": round(auc,4), "F1": round(f1_score(y_true,pred,zero_division=0),4)
}

# B7: Logistic Regression on full metric vector (5-fold CV)
X   = df_test[METRIC_COLS].values
lr  = LogisticRegression(max_iter=1000, random_state=42)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_cv = cross_val_score(lr, X, y_true, cv=cv, scoring="roc_auc").mean()
f1_cv  = cross_val_score(lr, X, y_true, cv=cv, scoring="f1").mean()
results["Logistic Regression"] = {"AUC-ROC": round(auc_cv,4), "F1": round(f1_cv,4)}

# B8: DAQS (our method)
daqs  = df_test["daqs"].values
auc   = roc_auc_score(y_true, daqs)
pred  = (daqs > np.median(daqs)).astype(int)
results["DAQS (Ours) ★"] = {
    "AUC-ROC": round(auc,4), "F1": round(f1_score(y_true,pred,zero_division=0),4)
}

df_results = pd.DataFrame(results).T.sort_values("AUC-ROC", ascending=False)
df_results.index.name = "Method"

print("\n📊 BASELINE COMPARISON (Natural vs. Synthetic classification):")
print(tabulate(df_results, headers=["Method"]+df_results.columns.tolist(),
               tablefmt="grid", floatfmt=".4f"))

df_results.to_csv(OUTPUT_DIR / "results" / "baseline_comparison.csv")
print("\n✅ Results saved")



📊 BASELINE COMPARISON (Natural vs. Synthetic classification):
+---------------------+-----------+--------+
| Method              |   AUC-ROC |     F1 |
+=====================+===========+========+
| MCD                 |    1.0000 | 0.6667 |
+---------------------+-----------+--------+
| PESQ                |    1.0000 | 0.6667 |
+---------------------+-----------+--------+
| Naive Average (μ)   |    1.0000 | 0.6667 |
+---------------------+-----------+--------+
| Logistic Regression |    1.0000 | 1.0000 |
+---------------------+-----------+--------+
| DAQS (Ours) ★       |    1.0000 | 0.6667 |
+---------------------+-----------+--------+
| UTMOS               |    0.6642 | 0.4733 |
+---------------------+-----------+--------+
| WER                 |    0.6008 | 0.4013 |
+---------------------+-----------+--------+
| DNSMOS              |    0.5621 | 0.3533 |
+---------------------+-----------+--------+

✅ Results saved


In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8.2 — Figure 6: ROC curves for all methods
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

plot_methods = {
    "MCD"              : df_test["mcd_n"].values,
    "PESQ"             : df_test["pesq_n"].values,
    "UTMOS"            : df_test["utmos_n"].values,
    "DNSMOS"           : df_test["dnsmos_n"].values,
    "WER"              : df_test["wer_n"].values,
    "Naive Average (μ)": df_test[METRIC_COLS].mean(axis=1).values,
    "DAQS (Ours) ★"   : df_test["daqs"].values,
}
linestyles = ["--","--","--","--","--","-.","-"]
roc_colors = ["#90A4AE","#78909C","#607D8B","#546E7A","#455A64","#FF9800","#D32F2F"]

for (name, s), ls, col in zip(plot_methods.items(), linestyles, roc_colors):
    fpr, tpr, _ = roc_curve(y_true, s)
    auc         = roc_auc_score(y_true, s)
    lw          = 2.8 if "DAQS" in name else 1.4
    ax.plot(fpr, tpr, ls=ls, color=col, linewidth=lw, label=f"{name}  (AUC={auc:.3f})")

ax.plot([0,1],[0,1],"k:",linewidth=0.8,label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Figure 6: ROC Curves — Natural vs. Synthetic Classification",
             fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig6_roc_curves.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 6 saved")


✅ Figure 6 saved


In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8.3 — Figure 7: AUC comparison bar chart
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

sorted_res = df_results.sort_values("AUC-ROC")
bar_colors = ["#D32F2F" if "DAQS" in idx else "#90A4AE" for idx in sorted_res.index]
bars = ax.barh(sorted_res.index, sorted_res["AUC-ROC"],
               color=bar_colors, edgecolor="black", linewidth=0.6)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Random baseline")
ax.set_xlabel("AUC-ROC (Natural vs. Synthetic)")
ax.set_title("Figure 7: Method Comparison by AUC-ROC", fontweight="bold")
ax.set_xlim([0.4, 1.06])

for bar, val in zip(bars, sorted_res["AUC-ROC"]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)

ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig7_auc_comparison.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 7 saved")


✅ Figure 7 saved


---
## Section 9: Statistical Significance Testing

**Test used:** Wilcoxon signed-rank test (paired on utterances) as required by rubric Section 3.2.  
Non-significant results (p ≥ 0.05) are explicitly reported — not hidden.


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.1 — Wilcoxon signed-rank tests: DAQS vs each baseline
# ─────────────────────────────────────────────────────────────────────────────
daqs_scores = df_test["daqs"].values
sig_rows    = []

# vs individual metrics
for col, name in METRIC_NAMES.items():
    baseline = df_test[col].values
    try:
        stat, p = wilcoxon(daqs_scores, baseline, alternative="two-sided",
                           zero_method="pratt")
        direction = "DAQS > baseline" if daqs_scores.mean() > baseline.mean() else "Baseline ≥ DAQS"
        sig_rows.append({
            "Comparison"   : f"DAQS vs {name}",
            "W-statistic"  : round(stat, 2),
            "p-value"      : round(p, 6),
            "Significant?" : "✅ p<0.05" if p < 0.05 else "❌ p≥0.05 (not significant)",
            "Direction"    : direction,
        })
    except ValueError as e:
        sig_rows.append({
            "Comparison"   : f"DAQS vs {name}",
            "W-statistic"  : "N/A",
            "p-value"      : "N/A",
            "Significant?" : f"Error: {e}",
            "Direction"    : "N/A",
        })

# vs naive average
naive = df_test[METRIC_COLS].mean(axis=1).values
try:
    stat, p   = wilcoxon(daqs_scores, naive, alternative="two-sided", zero_method="pratt")
    direction = "DAQS > Naive" if daqs_scores.mean() > naive.mean() else "Naive ≥ DAQS"
    sig_rows.append({
        "Comparison"   : "DAQS vs Naive Average",
        "W-statistic"  : round(stat, 2),
        "p-value"      : round(p, 6),
        "Significant?" : "✅ p<0.05" if p < 0.05 else "❌ p≥0.05 (not significant)",
        "Direction"    : direction,
    })
except ValueError as e:
    sig_rows.append({"Comparison":"DAQS vs Naive Average","Significant?":f"Error:{e}","Direction":"N/A","W-statistic":"N/A","p-value":"N/A"})

df_sig = pd.DataFrame(sig_rows)
print("📊 STATISTICAL SIGNIFICANCE — Wilcoxon Signed-Rank Test:")
print(tabulate(df_sig, headers="keys", tablefmt="grid", showindex=False))
df_sig.to_csv(OUTPUT_DIR / "results" / "significance_tests.csv", index=False)
print("\n✅ Significance tests saved")


📊 STATISTICAL SIGNIFICANCE — Wilcoxon Signed-Rank Test:
+-----------------------+---------------+-----------+---------------------------------------------------------------------------------------+-----------------+
| Comparison            | W-statistic   | p-value   | Significant?                                                                          | Direction       |
+=======================+===============+===========+=======================================================================================+=================+
| DAQS vs MCD           | 6102.0        | 0.0       | ✅ p<0.05                                                                             | DAQS > baseline |
+-----------------------+---------------+-----------+---------------------------------------------------------------------------------------+-----------------+
| DAQS vs PESQ          | 5748.0        | 0.0       | ✅ p<0.05                                                                             | DAQS

---
## Section 10: Ablation Study (Required by Rubric Section 3.2)

**Ablation:** Remove the disagreement penalty term (set λ=0), which reduces DAQS to the naive mean μ(u).  
This isolates the contribution of the disagreement term to classification performance.


In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10.1 — Ablation study: effect of removing the disagreement term
# ─────────────────────────────────────────────────────────────────────────────
ablation = {}

# Full DAQS (best λ)
daqs_full = compute_daqs(df_test["mu"].values, df_test["D"].values, best_lam)
auc_full  = roc_auc_score(y_true, daqs_full)
pred_full = (daqs_full > np.median(daqs_full)).astype(int)
ablation[f"DAQS Full (λ={best_lam})"] = {
    "λ"      : best_lam,
    "AUC-ROC": round(auc_full, 4),
    "F1"     : round(f1_score(y_true, pred_full, zero_division=0), 4),
    "Note"   : "Full model with disagreement penalty"
}

# Ablated: λ=0 → reduces to naive mean μ
mu_only  = df_test["mu"].values      # same as compute_daqs(..., lam=0)
auc_abl  = roc_auc_score(y_true, mu_only)
pred_abl = (mu_only > np.median(mu_only)).astype(int)
ablation["DAQS Ablated (λ=0 → Naive μ)"] = {
    "λ"      : 0.0,
    "AUC-ROC": round(auc_abl, 4),
    "F1"     : round(f1_score(y_true, pred_abl, zero_division=0), 4),
    "Note"   : "Disagreement term removed → reduces to naive average"
}

# Over-penalized: λ=2.0
daqs_ov  = compute_daqs(df_test["mu"].values, df_test["D"].values, 2.0)
auc_ov   = roc_auc_score(y_true, daqs_ov)
pred_ov  = (daqs_ov > np.median(daqs_ov)).astype(int)
ablation["DAQS Over-penalized (λ=2.0)"] = {
    "λ"      : 2.0,
    "AUC-ROC": round(auc_ov, 4),
    "F1"     : round(f1_score(y_true, pred_ov, zero_division=0), 4),
    "Note"   : "λ too large — over-penalizes mid-range utterances"
}

df_abl = pd.DataFrame(ablation).T
print("📊 ABLATION STUDY — Effect of λ (Disagreement Penalty):")
print(tabulate(df_abl, headers="keys", tablefmt="grid", floatfmt=".4f"))

auc_drop = auc_full - auc_abl
print(f"\n→ AUC-ROC drop when disagreement term removed: {auc_drop:+.4f}")
print(f"   ({'+' if auc_drop>=0 else ''}{'improvement' if auc_drop>0 else 'degradation'} from adding disagreement penalty)")
df_abl.to_csv(OUTPUT_DIR / "results" / "ablation_study.csv")
print("\n✅ Ablation study saved")


📊 ABLATION STUDY — Effect of λ (Disagreement Penalty):
+------------------------------+--------+-----------+--------+------------------------------------------------------+
|                              |      λ |   AUC-ROC |     F1 | Note                                                 |
+==============================+========+===========+========+======================================================+
| DAQS Full (λ=0.0)            | 0.0000 |    1.0000 | 0.6667 | Full model with disagreement penalty                 |
+------------------------------+--------+-----------+--------+------------------------------------------------------+
| DAQS Ablated (λ=0 → Naive μ) | 0.0000 |    1.0000 | 0.6667 | Disagreement term removed → reduces to naive average |
+------------------------------+--------+-----------+--------+------------------------------------------------------+
| DAQS Over-penalized (λ=2.0)  | 2.0000 |    1.0000 | 0.6667 | λ too large — over-penalizes mid-range utterances    |
+

In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10.2 — Figure 8: Ablation bar chart
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

variants   = list(ablation.keys())
aucs       = [ablation[v]["AUC-ROC"] for v in variants]
bar_colors = ["#4CAF50", "#F44336", "#FF9800"]

bars = ax.bar(range(len(variants)), aucs,
              color=bar_colors, edgecolor="black", linewidth=0.7)
ax.set_xticks(range(len(variants)))
ax.set_xticklabels(variants, rotation=12, ha="right", fontsize=9)
ax.set_ylabel("AUC-ROC")
ax.set_ylim([min(aucs) - 0.06, 1.04])
ax.set_title("Figure 8: Ablation Study — Contribution of Disagreement Term",
             fontweight="bold")
ax.grid(axis="y", alpha=0.25)

for bar, val in zip(bars, aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "fig8_ablation.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Figure 8 saved")


✅ Figure 8 saved


---
## Section 11: Dataset Documentation

> Required by Rubric Section 3.2 — Name, source, size, language, speakers, licence.

In [40]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11.1 — Dataset documentation table
# ─────────────────────────────────────────────────────────────────────────────
datasets = [
    {
        "Name"     : "LJSpeech-1.1",
        "Source"   : "keithito.com/LJ-Speech-Dataset",
        "Size"     : "13,100 utterances / ~24 hrs",
        "Language" : "English",
        "Speakers" : "1 (female)",
        "Licence"  : "Public Domain",
        "Role"     : "Natural speech — reference & TTS text prompts",
    },
    {
        "Name"     : "SpeedySpeech outputs (self-generated)",
        "Source"   : "Coqui TTS — github.com/coqui-ai/TTS",
        "Size"     : f"{N_UTTERANCES} utterances (generated from LJSpeech text)",
        "Language" : "English",
        "Speakers" : "1 (synthetic LJSpeech voice)",
        "Licence"  : "MPL-2.0 (open source)",
        "Role"     : "Low-quality / concatenative-proxy TTS system",
    },
    {
        "Name"     : "Tacotron2-DDC outputs (self-generated)",
        "Source"   : "Coqui TTS — github.com/coqui-ai/TTS",
        "Size"     : f"{N_UTTERANCES} utterances (generated from LJSpeech text)",
        "Language" : "English",
        "Speakers" : "1 (synthetic LJSpeech voice)",
        "Licence"  : "MPL-2.0 (open source)",
        "Role"     : "Neural TTS — medium quality",
    },
    {
        "Name"     : "VITS outputs (self-generated)",
        "Source"   : "Coqui TTS — github.com/coqui-ai/TTS",
        "Size"     : f"{N_UTTERANCES} utterances (generated from LJSpeech text)",
        "Language" : "English",
        "Speakers" : "1 (synthetic LJSpeech voice)",
        "Licence"  : "MPL-2.0 (open source)",
        "Role"     : "State-of-the-art TTS — highest quality tier",
    },
]

df_docs = pd.DataFrame(datasets)
print("📋 DATASET DOCUMENTATION:")
print(tabulate(df_docs, headers="keys", tablefmt="grid", showindex=False))
df_docs.to_csv(OUTPUT_DIR / "results" / "dataset_documentation.csv", index=False)
print("\n✅ Dataset documentation saved")


📋 DATASET DOCUMENTATION:
+----------------------------------------+-------------------------------------+-----------------------------------------------+------------+------------------------------+-----------------------+-----------------------------------------------+
| Name                                   | Source                              | Size                                          | Language   | Speakers                     | Licence               | Role                                          |
+========================================+=====================================+===============================================+============+==============================+=======================+===============================================+
| LJSpeech-1.1                           | keithito.com/LJ-Speech-Dataset      | 13,100 utterances / ~24 hrs                   | English    | 1 (female)                   | Public Domain         | Natural speech — reference & TTS text promp

---
## Section 12: Final Results Summary

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 12.1 — Complete project results summary
# ─────────────────────────────────────────────────────────────────────────────
line = "=" * 68
print(line)
print("  DAQS PROJECT — FINAL RESULTS")
print("  Student : Qamar Mehmood | Roll : 26I-7800")
print("  Topic   : T6 — Speech Quality Evaluation: Natural vs. Synthetic")
print(line)

# DAQS formula
print(f"\n  📐 DAQS FORMULA:")
print(f"     DAQS(u) = μ(u) - {best_lam} × D(u) × (1 - μ(u))")

# Calibration
print(f"\n  ⚙️  CALIBRATION:")
print(f"     Best λ = {best_lam}  |  AUC on calibration set = {best_auc:.4f}")
print(f"     Disagreement threshold θ = {theta:.4f}")
total_high = df_norm.high_disagreement.sum()
print(f"     High-disagreement utterances = {total_high} ({total_high/len(df_norm)*100:.1f}%)")

# Top results
print(f"\n  📊 TOP CLASSIFICATION RESULTS (AUC-ROC):")
for idx, row in df_results.head(5).iterrows():
    mark = " ◄ OUR METHOD" if "DAQS" in idx else ""
    print(f"     {idx:<35}  AUC={row['AUC-ROC']:.4f}  F1={row['F1']:.4f}{mark}")

# Ablation
drop = auc_full - auc_abl
print(f"\n  🔬 ABLATION:")
print(f"     AUC drop when disagreement term removed: {drop:+.4f}")
print(f"     {'Disagreement term HELPS classification' if drop > 0 else 'Disagreement term does not improve on this data'}")

# Error types
if len(error_counts) > 0:
    print(f"\n  🏷️  ERROR TYPES (high-disagreement utterances):")
    for et, cnt in error_counts.items():
        print(f"     {et}: {cnt} utterances ({cnt/len(high_d)*100:.1f}%)")

# Output files
print(f"\n  📁 ALL OUTPUT FILES:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"     {str(f.relative_to(OUTPUT_DIR)):45s}  {size_kb:7.1f} KB")

print()
print(line)
print("  All figures are reproducible from this notebook.")
print("  For the paper: use df_results, df_sig, df_abl, df_docs tables.")
print(line)


  DAQS PROJECT — FINAL RESULTS
  Student : Qamar Mehmood | Roll : 26I-7800
  Topic   : T6 — Speech Quality Evaluation: Natural vs. Synthetic

  📐 DAQS FORMULA:
     DAQS(u) = μ(u) - 0.0 × D(u) × (1 - μ(u))

  ⚙️  CALIBRATION:
     Best λ = 0.0  |  AUC on calibration set = 1.0000
     Disagreement threshold θ = 0.4700
     High-disagreement utterances = 58 (14.5%)

  📊 TOP CLASSIFICATION RESULTS (AUC-ROC):
     MCD                                  AUC=1.0000  F1=0.6667
     PESQ                                 AUC=1.0000  F1=0.6667
     Naive Average (μ)                    AUC=1.0000  F1=0.6667
     Logistic Regression                  AUC=1.0000  F1=1.0000
     DAQS (Ours) ★                        AUC=1.0000  F1=0.6667 ◄ OUR METHOD

  🔬 ABLATION:
     AUC drop when disagreement term removed: +0.0000
     Disagreement term does not improve on this data

  🏷️  ERROR TYPES (high-disagreement utterances):
     Type B: Intelligibility-Quality Conflict: 42 utterances (72.4%)
     Type A: Spe